# 24 — Consensus, Raft & Failure Models

### Why this exists
Consensus is about agreeing on a value despite failures. This lab models terms, votes, quorum and leader failure without pretending the toy is a full Raft implementation.

### Learning objectives
reason about terms and quorum; simulate elections; distinguish availability from safety

### Prerequisite
Python sets and state machines

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/24-consensus-raft-failure-models.svg)

> **Question to carry through the lab:** Why is a majority needed before a leader can safely claim authority? What happens when the network splits?

## Capability contract

**Capability:** C05 — Storage Consistency & Distributed Systems  
**Workstream:** Distributed Data Platform  
**Primary roles:** Data Engineer|Platform Engineer|Architect  
**You will prove:** Explain a distributed trade-off


## Mental model
Raft uses replicated state, terms, elections and majority agreement. A partition may reduce availability, but safety depends on quorum rules and stale leaders losing authority.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S08 — Replication and consistency (DATA-1110)** → [`../work_simulations/08_acme_commerce_replica_lag/README.md`](../work_simulations/08_acme_commerce_replica_lag/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
nodes={"a","b","c","d","e"}
term=1
votes={"a","b","c"}
quorum=len(nodes)//2+1
print({"quorum":quorum,"votes":len(votes),"leader_elected":len(votes)>=quorum})

In [ ]:
# Election attempt with a stale node excluded.
def elect(votes,total): return len(votes)>=total//2+1
for votes in [{"a","b"},{"a","b","c"},{"a","d"}]: print(votes,elect(votes,len(nodes)))

In [ ]:
# Network partition: 2 vs 3. Only the majority side can form a quorum.
majority={"a","b","c"}; minority={"d","e"}
print("majority quorum",elect(majority,len(nodes)))
print("minority quorum",elect(minority,len(nodes)))

In [ ]:
# Break: allow a leader to act without quorum and make the unsafe assumption explicit.
def unsafe_write(active): return bool(active)
print("minority could write only if quorum enforcement is removed:",unsafe_write(minority))
print("safe rule: commit requires quorum")

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## 8. Production bridge
Production consensus requires persistent terms/logs, leader completeness, election timeouts, membership changes and rigorous failure semantics. This lab intentionally models only the core intuition.

## 9. Independent challenge
Add message loss and a stale term. Decide which state transitions are safe and which are only liveness failures. Document the failure model you are actually simulating.

### Evidence to keep
quorum calculation, election outcomes, partition evidence, and a clear safety-vs-availability explanation

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.